# 01 - Entendimento dos Dados

Objetivo: carregar os arquivos brutos do Kaggle, validar disponibilidade, unir as fontes e inspecionar qualidade básica antes de modelar.

Boas práticas usadas aqui:

- Reutilizar o código de produção em `src/`.
- Não criar features de modelo nesta etapa.
- Confirmar distribuição da variável alvo e cobertura temporal.
- Evitar usar acurácia como referência de performance.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/developer/workspace_python/financial_transactions_pipeline')

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.settings import Settings
from src.data.load_data import RawDataRepository
from src.data.merge_data import FraudDataMerger
from src.features.cleaning import FraudDataCleaner

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

settings = Settings(project_root=PROJECT_ROOT)
settings.raw_data_dir

WindowsPath('D:/developer/workspace_python/financial_transactions_pipeline/data/raw')

## 1. Verificar arquivos esperados

Baixe o dataset do Kaggle e coloque os arquivos em `data/raw`.

In [3]:
expected_files = [
    "transactions_data.csv",
    "cards_data.csv",
    "users_data.csv",
    "mcc_codes.json",
    "train_fraud_labels.json",
]

file_status = pd.DataFrame(
    {
        "file": expected_files,
        "exists": [(settings.raw_data_dir / name).exists() for name in expected_files],
        "path": [str(settings.raw_data_dir / name) for name in expected_files],
    }
)
file_status

,file,exists,path
0,transactions_data.csv,False,D:\developer\workspace_python\financial_transa...
1,cards_data.csv,False,D:\developer\workspace_python\financial_transa...
2,users_data.csv,False,D:\developer\workspace_python\financial_transa...
3,mcc_codes.json,False,D:\developer\workspace_python\financial_transa...
4,train_fraud_labels.json,False,D:\developer\workspace_python\financial_transa...


## 2. Carregar fontes brutas

In [4]:
repo = RawDataRepository(settings)
raw = repo.load_all()

shape_summary = []
for name, value in raw.items():
    if isinstance(value, pd.DataFrame):
        shape_summary.append({"source": name, "rows": value.shape[0], "columns": value.shape[1]})
    else:
        shape_summary.append({"source": name, "rows": len(value) if hasattr(value, "__len__") else None, "columns": None})

pd.DataFrame(shape_summary)

2026-07-12 21:55:50 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/transactions_data.csv
2026-07-12 21:56:57 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/cards_data.csv
2026-07-12 21:56:58 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/users_data.csv
2026-07-12 21:56:58 | INFO | src.data.load_data | Carregando JSON de minio: data/raw/mcc_codes.json
2026-07-12 21:56:58 | INFO | src.data.load_data | Carregando JSON de minio: data/raw/train_fraud_labels.json


,source,rows,columns
0,transactions,13305915,12.0
1,cards,6146,13.0
2,users,2000,14.0
3,mcc,109,NaN
4,labels,1,NaN


In [5]:
raw["transactions"].head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [6]:
raw["cards"].head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [7]:
raw["users"].head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [8]:
raw["mcc"]

{'5812': 'Eating Places and Restaurants',
 '5541': 'Service Stations',
 '7996': 'Amusement Parks, Carnivals, Circuses',
 '5411': 'Grocery Stores, Supermarkets',
 '4784': 'Tolls and Bridge Fees',
 '4900': 'Utilities - Electric, Gas, Water, Sanitary',
 '5942': 'Book Stores',
 '5814': 'Fast Food Restaurants',
 '4829': 'Money Transfer',
 '5311': 'Department Stores',
 '5211': 'Lumber and Building Materials',
 '5310': 'Discount Stores',
 '3780': 'Computer Network Services',
 '5499': 'Miscellaneous Food Stores',
 '4121': 'Taxicabs and Limousines',
 '5300': 'Wholesale Clubs',
 '5719': 'Miscellaneous Home Furnishing Stores',
 '7832': 'Motion Picture Theaters',
 '5813': 'Drinking Places (Alcoholic Beverages)',
 '4814': 'Telecommunication Services',
 '5661': 'Shoe Stores',
 '5977': 'Cosmetic Stores',
 '8099': 'Medical Services',
 '7538': 'Automotive Service Shops',
 '5912': 'Drug Stores and Pharmacies',
 '4111': 'Local and Suburban Commuter Transportation',
 '5815': 'Digital Goods - Media, Books,

In [9]:
raw["labels"]

{'target': {'10649266': 'No',
  '23410063': 'No',
  '9316588': 'No',
  '12478022': 'No',
  '9558530': 'No',
  '12532830': 'No',
  '19526714': 'No',
  '9906964': 'No',
  '13224888': 'No',
  '13749094': 'No',
  '12303776': 'No',
  '19480376': 'No',
  '11716050': 'No',
  '20025400': 'No',
  '7661688': 'No',
  '16662807': 'No',
  '21419778': 'No',
  '18011186': 'No',
  '23289598': 'No',
  '11644547': 'No',
  '23235120': 'No',
  '19748218': 'No',
  '8720720': 'No',
  '18335831': 'No',
  '18936727': 'No',
  '15223870': 'No',
  '12370203': 'No',
  '17126661': 'No',
  '22270430': 'No',
  '18790248': 'No',
  '20143410': 'No',
  '9497252': 'No',
  '17619208': 'No',
  '11052664': 'No',
  '14670204': 'No',
  '17681877': 'No',
  '22485981': 'No',
  '22332853': 'No',
  '16628447': 'No',
  '7766832': 'No',
  '7614276': 'No',
  '14069486': 'No',
  '13755628': 'No',
  '17306332': 'No',
  '19822702': 'No',
  '19118845': 'No',
  '12799754': 'No',
  '17368331': 'No',
  '23652500': 'No',
  '14024256': 'No'

In [10]:
raw["transactions"].isnull().sum()

id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                1652706
mcc                      0
errors            13094522
dtype: int64

In [11]:
from data_profiling import ProfileReport

In [12]:
# 2. Cria o relatório
profile = ProfileReport(raw["transactions"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
profile.to_file("reports/relatorio_perfilamento.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [05:51<00:00, 29.31s/it]  


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
card = ProfileReport(raw["cards"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
card.to_file("reports/relatorio_perfilamento_cartao.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 42.40it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
users = ProfileReport(raw["users"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
users.to_file("reports/relatorio_perfilamento_users.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:00<00:00, 64.33it/s][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
mcc_df = pd.DataFrame(list(raw["mcc"].items()), columns=['mcc_code', 'mcc_description'])

mcc = ProfileReport(mcc_df, title="Relatório de Análise Exploratória", explorative=True)

# 7. Exporta como arquivo HTML para visualização
mcc.to_file("reports/relatorio_perfilamento_mcc.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 2/2 [00:00<00:00, 68.67it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
labels_df = pd.DataFrame(list(raw["labels"]['target'].items()), columns=['transaction_id', 'target'])

labels = ProfileReport(labels_df, title="Relatório de Análise Exploratória", explorative=True)

# 7. Exporta como arquivo HTML para visualização
labels.to_file("reports/relatorio_perfilamento_labels.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 2/2 [02:47<00:00, 83.91s/it] 


In [20]:
type(raw["labels"])

dict

In [ ]:
labels_df

In [ ]:
merged = FraudDataMerger(settings).merge(
    transactions=raw["transactions"],
    cards=raw["cards"],
    users=raw["users"],
    mcc_codes=raw["mcc"],
    labels=raw["labels"],
)
cleaned = FraudDataCleaner(settings).fit_transform(merged)

cleaned.shape, cleaned.head()